<a href="https://colab.research.google.com/github/Shivangi917/curvetopia-files/blob/main/audio_detector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
!pip install librosa pydub python_speech_features SpeechRecognition

In [ ]:
from google.colab import drive
import os
import pandas as pd

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Define dataset paths
confident_path = '/content/drive/MyDrive/Voicedata/confident'
non_confident_path = '/content/drive/MyDrive/Voicedata/Non-confident'

# List audio files
confident_files = [os.path.join(confident_path, f) for f in os.listdir(confident_path) if f.endswith('.wav')]
non_confident_files = [os.path.join(non_confident_path, f) for f in os.listdir(non_confident_path) if f.endswith('.wav')]

print(f"Found {len(confident_files)} confident samples and {len(non_confident_files)} non-confident samples.")

Found 473 confident samples and 528 non-confident samples.


In [ ]:
import librosa
import numpy as np
import speech_recognition as sr
from pydub import AudioSegment
from pydub.silence import detect_nonsilent

def extract_features(file_path):
    y, sr_audio = librosa.load(file_path, sr=None) # Changed sr to sr_audio

    # 1. Pitch (F0)
    pitches, _ = librosa.piptrack(y=y, sr=sr_audio) # Changed sr to sr_audio
    pitches = pitches[pitches > 0]
    pitch_mean = np.mean(pitches)
    pitch_std = np.std(pitches)

    # 2. Speech rate (words/sec)
    recognizer = sr.Recognizer()
    with sr.AudioFile(file_path) as source:
        audio = recognizer.record(source)
        try:
            text = recognizer.recognize_google(audio)
            words = text.split()
            duration = librosa.get_duration(y=y, sr=sr_audio) # Changed sr to sr_audio
            speech_rate = len(words) / duration if duration > 0 else 0
        except:
            speech_rate = 0

    # 3. Pause analysis
    audio_pydub = AudioSegment.from_wav(file_path)
    non_silent = detect_nonsilent(audio_pydub, min_silence_len=200, silence_thresh=-40)
    pauses = [(non_silent[i][0] - non_silent[i-1][1]) / 1000 for i in range(1, len(non_silent))]
    avg_pause = np.mean(pauses) if pauses else 0

    # 4. Volume (RMS) variability
    rms = librosa.feature.rms(y=y)[0]
    volume_std = np.std(rms)

    # 5. MFCCs (13 coefficients)
    mfccs = np.mean(librosa.feature.mfcc(y=y, sr=sr_audio, n_mfcc=13), axis=1) # Changed sr to sr_audio

    return {
        'pitch_mean': pitch_mean,
        'pitch_std': pitch_std,
        'speech_rate': speech_rate,
        'avg_pause': avg_pause,
        'volume_std': volume_std,
        **{f'mfcc_{i}': mfccs[i] for i in range(13)}
    }

# Extract features for all files
features = []
labels = []

print("Extracting features for Confident speakers...")
for i, file in enumerate(confident_files):
    print(f"\rProcessing confident file {i+1}/{len(confident_files)}: {file}", end="", flush=True)
    features.append(extract_features(file))
    labels.append(1)  # 1 = Confident
print("\nConfident files processed!")

print("\nExtracting features for Non-Confident speakers...")
for i, file in enumerate(non_confident_files):
    print(f"\rProcessing non-confident file {i+1}/{len(non_confident_files)}: {file}", end="", flush=True)
    features.append(extract_features(file))
    labels.append(0)  # 0 = Non-Confident
print("\nNon-Confident files processed!")

print("\nFeature extraction complete!")

Extracting features for Confident speakers...
Processing confident file 473/473: /content/drive/MyDrive/Voicedata/confident/97-1.wav
Confident files processed!

Extracting features for Non-Confident speakers...
Processing non-confident file 313/528: /content/drive/MyDrive/Voicedata/Non-confident/349-0.wav

/usr/local/lib/python3.11/dist-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=0
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/numpy/_core/fromnumeric.py:3596: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr/local/lib/python3.11/dist-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in divide
  ret = ret.dtype.type(ret / rcount)
/usr/local/lib/python3.11/dist-packages/numpy/_core/_methods.py:218: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/usr/local/lib/python3.11/dist-packages/numpy/_core/_methods.py:175: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
/usr/local/lib/python3.11/dist-packages/numpy/_core/_methods.py:210: RuntimeWarning: invalid value encountered in divide
  ret = ret.dtype.type(ret / rcount)


Processing non-confident file 528/528: /content/drive/MyDrive/Voicedata/Non-confident/Be confident don't feel shy #sandeepmaheshwari  #shorts (2).wav
Non-Confident files processed!

Feature extraction complete!


In [ ]:
import os

# Define the full path in your Google Drive
csv_save_path = '/content/drive/MyDrive/audio_confidence/confidence_dataset.csv'

# Convert and save
df = pd.DataFrame(features, index=range(len(features))) # Added index
df['label'] = labels
df.to_csv(csv_save_path, index=False)

print(f"Dataset saved to: {csv_save_path}")

Dataset saved to: /content/drive/MyDrive/audio_confidence/confidence_dataset.csv


In [ ]:
# Improved model comparison pipeline with tuning, stacking, and metrics

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.metrics import accuracy_score, classification_report
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.base import clone
import joblib

# Load dataset
csv_path = '/content/drive/MyDrive/audio_confidence/confidence_dataset.csv'
df = pd.read_csv(csv_path)

print("Missing values before:", df.isna().sum().sum())
df = df.dropna()
print("Missing values after:", df.isna().sum().sum())

X = df.drop('label', axis=1)
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"\n📊 Using {len(X_train)} training samples")
print(f"📊 Using {len(X_test)} test samples\n")

# Define base pipelines
base_preprocessing = [
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('feature_selector', SelectKBest(mutual_info_classif, k=10))
]

models = {
    "Random Forest": RandomForestClassifier(n_estimators=200, max_depth=7, min_samples_split=5, min_samples_leaf=3, class_weight='balanced', random_state=42),
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight='balanced', solver='liblinear'),
    "SVM": SVC(kernel='rbf', probability=True, class_weight='balanced'),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3),
    "KNN": KNeighborsClassifier(n_neighbors=7)
}

results = {}

for name, model in models.items():
    print(f"🔍 Evaluating: {name}")
    pipeline = Pipeline(base_preprocessing + [('classifier', model)])

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='accuracy')
    pipeline.fit(X_train, y_train)

    train_acc = accuracy_score(y_train, pipeline.predict(X_train))
    test_acc = accuracy_score(y_test, pipeline.predict(X_test))

    print(f"✅ CV Accuracy: {np.mean(cv_scores):.2f} ± {np.std(cv_scores):.2f}")
    print(f"✅ Train Accuracy: {train_acc:.2f}")
    print(f"✅ Test Accuracy: {test_acc:.2f}\n")

    results[name] = {
        'model': pipeline,
        'cv_mean': np.mean(cv_scores),
        'train_acc': train_acc,
        'test_acc': test_acc
    }

# Stacking ensemble
estimators = [(name.replace(" ", "_"), clone(results[name]['model'].named_steps['classifier'])) for name in results if name != 'Logistic Regression']
stk_model = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(),
    passthrough=True
)
stack_pipeline = Pipeline(base_preprocessing + [('classifier', stk_model)])
stack_pipeline.fit(X_train, y_train)

train_acc = accuracy_score(y_train, stack_pipeline.predict(X_train))
test_acc = accuracy_score(y_test, stack_pipeline.predict(X_test))
print("🔀 Evaluating: Stacking Ensemble")
print(f"✅ Train Accuracy: {train_acc:.2f}")
print(f"✅ Test Accuracy: {test_acc:.2f}\n")

results['Stacking Ensemble'] = {
    'model': stack_pipeline,
    'train_acc': train_acc,
    'test_acc': test_acc
}

# Best model
best_model_name = max(results, key=lambda x: results[x]['test_acc'])
best_model = results[best_model_name]['model']

print(f"🏆 Best Model: {best_model_name}")
print(f"Test Accuracy: {results[best_model_name]['test_acc']:.2f}")
print(f"Train Accuracy: {results[best_model_name]['train_acc']:.2f}")

model_path = f"/content/drive/MyDrive/audio_confidence/best_model_{best_model_name.replace(' ', '_')}.joblib"
joblib.dump(best_model, model_path)
print(f"📁 Saved best model to: {model_path}")

# Final report
y_pred_final = best_model.predict(X_test)
print("\n📄 Classification Report:\n", classification_report(y_test, y_pred_final))

Missing values before: 0
Missing values after: 0

📊 Using 810 training samples
📊 Using 203 test samples

🔍 Evaluating: Random Forest
✅ CV Accuracy: 0.72 ± 0.05
✅ Train Accuracy: 0.92
✅ Test Accuracy: 0.73

🔍 Evaluating: Logistic Regression
✅ CV Accuracy: 0.60 ± 0.03
✅ Train Accuracy: 0.62
✅ Test Accuracy: 0.59

🔍 Evaluating: SVM
✅ CV Accuracy: 0.69 ± 0.04
✅ Train Accuracy: 0.76
✅ Test Accuracy: 0.72

🔍 Evaluating: Gradient Boosting
✅ CV Accuracy: 0.71 ± 0.04
✅ Train Accuracy: 0.93
✅ Test Accuracy: 0.71

🔍 Evaluating: KNN
✅ CV Accuracy: 0.74 ± 0.04
✅ Train Accuracy: 0.85
✅ Test Accuracy: 0.75

🔀 Evaluating: Stacking Ensemble
✅ Train Accuracy: 0.88
✅ Test Accuracy: 0.76

🏆 Best Model: Stacking Ensemble
Test Accuracy: 0.76
Train Accuracy: 0.88
📁 Saved best model to: /content/drive/MyDrive/audio_confidence/best_model_Stacking_Ensemble.joblib

📄 Classification Report:
               precision    recall  f1-score   support

           0       0.77      0.78      0.77       107
           1  

In [27]:
import os
from google.colab import drive
import joblib

# Define the path to your saved files
drive_path = '/content/drive/MyDrive/audio_confidence'

# Load the model and scaler
model = joblib.load(os.path.join(drive_path, 'best_model_Stacking_Ensemble.joblib'))
scaler = joblib.load(os.path.join(drive_path, 'best_model_Stacking_Ensemble.joblib'))

print("Model and scaler loaded successfully!")


Model and scaler loaded successfully!


In [ ]:
from google.colab import files
import os
from pydub import AudioSegment
import librosa
import numpy as np
import speech_recognition as sr
from pydub.silence import detect_nonsilent
import pandas as pd
import joblib

# Function to convert audio to WAV format if needed
def convert_to_wav(input_path, output_path):
    audio = AudioSegment.from_file(input_path)
    audio.export(output_path, format="wav")
    return output_path

# Function to extract features (same as during training)
def extract_features(file_path):
    y, sr_audio = librosa.load(file_path, sr=None)

    # 1. Pitch (F0)
    pitches, _ = librosa.piptrack(y=y, sr=sr_audio)
    pitches = pitches[pitches > 0]
    pitch_mean = np.mean(pitches) if len(pitches) > 0 else 0
    pitch_std = np.std(pitches) if len(pitches) > 0 else 0

    # 2. Speech rate (words/sec)
    recognizer = sr.Recognizer()
    with sr.AudioFile(file_path) as source:
        audio = recognizer.record(source)
        try:
            text = recognizer.recognize_google(audio)
            words = text.split()
            duration = librosa.get_duration(y=y, sr=sr_audio)
            speech_rate = len(words) / duration if duration > 0 else 0
        except:
            speech_rate = 0

    # 3. Pause analysis
    audio_pydub = AudioSegment.from_wav(file_path)
    non_silent = detect_nonsilent(audio_pydub, min_silence_len=200, silence_thresh=-40)
    pauses = [(non_silent[i][0] - non_silent[i-1][1]) / 1000 for i in range(1, len(non_silent))]
    avg_pause = np.mean(pauses) if pauses else 0

    # 4. Volume (RMS) variability
    rms = librosa.feature.rms(y=y)[0]
    volume_std = np.std(rms) if len(rms) > 0 else 0

    # 5. MFCCs (13 coefficients)
    mfccs = np.mean(librosa.feature.mfcc(y=y, sr=sr_audio, n_mfcc=13), axis=1)

    return {
        'pitch_mean': pitch_mean,
        'pitch_std': pitch_std,
        'speech_rate': speech_rate,
        'avg_pause': avg_pause,
        'volume_std': volume_std,
        **{f'mfcc_{i}': mfccs[i] for i in range(13)}
    }

# Mount Google Drive (where your model is stored)
from google.colab import drive
drive.mount('/content/drive')

# Load the trained model
model_path = '/content/drive/MyDrive/audio_confidence/best_model_Stacking_Ensemble.joblib'
model = joblib.load(model_path)

# Upload audio file
uploaded = files.upload()
file_name = next(iter(uploaded))
file_path = f"/content/{file_name}"

# Check if file needs conversion
if not file_name.lower().endswith('.wav'):
    print(f"Converting {file_name} to WAV format...")
    wav_path = f"/content/{os.path.splitext(file_name)[0]}.wav"
    file_path = convert_to_wav(file_path, wav_path)
    print(f"Converted to: {wav_path}")

# Extract features from the audio file
print("\nExtracting features from audio file...")
features = extract_features(file_path)

# Convert features to DataFrame
features_df = pd.DataFrame([features])

# Make prediction
prediction = model.predict(features_df)
confidence = model.predict_proba(features_df)

# Display results
class_map = {0: "Non-Confident", 1: "Confident"}
print("\nPrediction Results:")
print(f"Class: {class_map[prediction[0]]}")
print(f"Confidence Scores: Non-Confident: {confidence[0][0]:.2f}, Confident: {confidence[0][1]:.2f}")

# Clean up temporary files
if os.path.exists(file_path):
    os.remove(file_path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Saving 04-1.wav to 04-1.wav

Extracting features from audio file...

Prediction Results:
Class: Confident
Confidence Scores: Non-Confident: 0.22, Confident: 0.78


In [ ]:
import os
from google.colab import files
import pandas as pd
from shutil import move
from pathlib import Path

csv_path = '/content/drive/MyDrive/audio_confidence/confidence_dataset.csv'

def handle_upload_and_extract(label, save_path, label_value):
    print(f"Please upload {label} audio files:")
    uploaded = files.upload()
    new_data = []

    for fname in uploaded.keys():
        base, ext = os.path.splitext(fname)

        # Convert to .wav if needed
        if ext.lower() != '.wav':
            audio = AudioSegment.from_file(fname)
            fname_wav = base + '.wav'
            audio.export(fname_wav, format='wav')
            os.remove(fname)
        else:
            fname_wav = fname

        # Move to appropriate folder
        dest_path = os.path.join(save_path, os.path.basename(fname_wav))
        move(fname_wav, dest_path)

        # Extract features
        features = extract_features(dest_path)
        features['label'] = label_value
        new_data.append(features)

    # Append to CSV
    if new_data:
        df_new = pd.DataFrame(new_data)
        if os.path.exists(csv_path):
            df_existing = pd.read_csv(csv_path)
            df_combined = pd.concat([df_existing, df_new], ignore_index=True)
        else:
            df_combined = df_new
        df_combined.to_csv(csv_path, index=False)
        print(f"{label} audio features added to dataset.")

# Ask for confident audios
add_confident = input("Do you have confident audio files to add? (yes/no): ").strip().lower()
if add_confident == 'yes':
    handle_upload_and_extract("confident", confident_path, label_value=1)

# Ask for non-confident audios
add_non_confident = input("Do you have non-confident audio files to add? (yes/no): ").strip().lower()
if add_non_confident == 'yes':
    handle_upload_and_extract("non-confident", non_confident_path, label_value=0)


Do you have confident audio files to add? (yes/no): no
Do you have non-confident audio files to add? (yes/no): yes
Please upload non-confident audio files:


In [16]:
!pip install flask flask-jwt-extended librosa pydub speechrecognition scikit-learn pandas numpy joblib

In [17]:
!pip install flask-cors

In [10]:
from flask_cors import CORS

# Initialize Flask app
app = Flask(__name__)

# Enable CORS for all routes (during development)
CORS(app)

In [18]:
!pip install pyngrok ngrok

In [30]:
from pyngrok import ngrok
!ngrok authtoken your-auth-token

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [21]:
# Install system dependency
!apt-get install ffmpeg -y

# Install Python libraries
!pip install flask pydub librosa numpy SpeechRecognition pandas joblib moviepy pyngrok -q

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 34 not upgraded.


In [32]:
from flask import Flask, request, jsonify
from flask_cors import CORS
import os
from pydub import AudioSegment
import numpy as np
import pandas as pd
import librosa
import speech_recognition as sr
from pydub.silence import detect_nonsilent
import joblib
from werkzeug.utils import secure_filename
from google.colab import drive
from pyngrok import ngrok
import traceback
import warnings

# Suppress warnings
warnings.filterwarnings("ignore", category=FutureWarning)

# Install dependencies (run in Colab only)
!pip install flask-cors pyngrok pydub ffmpeg-python -q
!apt install ffmpeg -y

# Initialize Flask app
app = Flask(__name__)
CORS(app)  # Enable CORS for all origins

# Configure upload folder
UPLOAD_FOLDER = '/content/uploads'
os.makedirs(UPLOAD_FOLDER, exist_ok=True)
app.config['UPLOAD_FOLDER'] = UPLOAD_FOLDER
ALLOWED_EXTENSIONS = {'wav', 'mp3', 'ogg', 'm4a', 'webm'}

# Mount Google Drive and load model
drive.mount('/content/drive', force_remount=True)
model_path = '/content/drive/MyDrive/audio_confidence/best_model_Stacking_Ensemble.joblib'
model = joblib.load(model_path)

# Utility functions
def allowed_file(filename):
    return '.' in filename and filename.rsplit('.', 1)[1].lower() in ALLOWED_EXTENSIONS

def convert_to_wav(input_path, output_path):
    try:
        audio = AudioSegment.from_file(input_path)
        audio = audio.set_frame_rate(16000).set_channels(1)
        # Explicitly specify sample format for PCM WAV
        audio.export(output_path, format="wav", codec="pcm_s16le")
        return output_path
    except Exception as e:
        print(f"Conversion error: {str(e)}")
        traceback.print_exc()
        return None

def extract_features(file_path):
    try:
        # Try loading with different backends
        try:
            y, sr_audio = librosa.load(file_path, sr=None, res_type='kaiser_fast')
        except Exception as e:
            print(f"Primary load failed, trying fallback: {str(e)}")
            y, sr_audio = librosa.load(file_path, sr=None, mono=True)

        # 1. Pitch (F0)
        pitches, _ = librosa.piptrack(y=y, sr=sr_audio)
        pitches = pitches[pitches > 0]
        pitch_mean = np.mean(pitches) if len(pitches) > 0 else 0
        pitch_std = np.std(pitches) if len(pitches) > 0 else 0

        # 2. Speech rate
        recognizer = sr.Recognizer()
        with sr.AudioFile(file_path) as source:
            audio_data = recognizer.record(source)
            try:
                text = recognizer.recognize_google(audio_data)
                words = text.split()
                duration = librosa.get_duration(y=y, sr=sr_audio)
                speech_rate = len(words) / duration if duration > 0 else 0
            except Exception as e:
                print(f"Speech recognition error: {str(e)}")
                speech_rate = 0

        # 3. Pause analysis
        try:
            audio_pydub = AudioSegment.from_wav(file_path)
            non_silent = detect_nonsilent(audio_pydub, min_silence_len=200, silence_thresh=-40)
            pauses = [(non_silent[i][1] - non_silent[i][0]) / 1000 for i in range(len(non_silent)-1)]
            avg_pause = np.mean(pauses) if pauses else 0
        except Exception as e:
            print(f"Pause analysis error: {str(e)}")
            avg_pause = 0

        # 4. Volume variability
        rms = librosa.feature.rms(y=y)[0]
        volume_std = np.std(rms) if len(rms) > 0 else 0

        # 5. MFCCs
        mfccs = np.mean(librosa.feature.mfcc(y=y, sr=sr_audio, n_mfcc=13), axis=1)

        return {
            'pitch_mean': pitch_mean,
            'pitch_std': pitch_std,
            'speech_rate': speech_rate,
            'avg_pause': avg_pause,
            'volume_std': volume_std,
            **{f'mfcc_{i}': mfccs[i] for i in range(13)}
        }
    except Exception as e:
        print(f"Error processing file: {str(e)}")
        traceback.print_exc()
        return None

# Routes
@app.route('/api/predict', methods=['POST'])
def predict():
    if 'file' not in request.files:
        return jsonify({'error': 'No file part', 'status': 400}), 400

    file = request.files['file']
    if file.filename == '':
        return jsonify({'error': 'No selected file', 'status': 400}), 400

    if not allowed_file(file.filename):
        return jsonify({'error': 'File type not allowed', 'status': 400}), 400

    try:
        filename = secure_filename(file.filename)
        upload_path = os.path.join(app.config['UPLOAD_FOLDER'], filename)
        file.save(upload_path)

        # Always convert to WAV to ensure consistent format
        wav_filename = f"converted_{os.path.splitext(filename)[0]}.wav"
        wav_path = os.path.join(app.config['UPLOAD_FOLDER'], wav_filename)

        if not convert_to_wav(upload_path, wav_path):
            os.remove(upload_path)
            return jsonify({'error': 'File conversion failed', 'status': 500}), 500

        features = extract_features(wav_path)
        if features is None:
            os.remove(upload_path)
            os.remove(wav_path)
            return jsonify({'error': 'Error processing audio file', 'status': 500}), 500

        features_df = pd.DataFrame([features])
        prediction = model.predict(features_df)[0]
        confidence = model.predict_proba(features_df)[0]

        # Clean up files
        os.remove(upload_path)
        os.remove(wav_path)

        return jsonify({
            'status': 200,
            'filename': filename,
            'prediction': 'Confident' if prediction == 1 else 'Non-Confident',
            'confidence': {
                'confident': float(confidence[1]),
                'non_confident': float(confidence[0])
            },
            'features': {k: float(v) for k, v in features.items()}
        })

    except Exception as e:
        if os.path.exists(upload_path):
            os.remove(upload_path)
        if os.path.exists(wav_path):
            os.remove(wav_path)
        return jsonify({'error': str(e), 'status': 500}), 500

@app.route('/')
def health_check():
    return jsonify({'status': 'healthy', 'message': 'API is running'})

# Start Flask + ngrok
if __name__ == '__main__':
    public_url = ngrok.connect(5000)
    print(" * Public URL:", public_url)
    app.run(host='0.0.0.0', port=5000)

^C
Mounted at /content/drive
 * Public URL: NgrokTunnel: "https://7d0c-35-245-151-123.ngrok-free.app" -> "http://localhost:5000"
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit
